# ⚠️ TEST — Project 1 applied to UCI Adult

**Experimental public-data application.** The Project 1 fairness audit (base rates, demographic parity, equalized odds, calibration, impossibility demo) applied to a canonical public dataset.

* **Data:** UCI Adult (income), https://archive.ics.uci.edu/ml/datasets/adult
* **Task:** predict `income > 50K` with a logistic GLM (the "risk model")
* **Protected attributes:** sex, race
* **Note:** this is a *demonstration of the pipeline*, not a claim about the real world — the model includes protected attributes as predictors on purpose so the fairness metrics have something to measure.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT.parent))  # repo root for src.fairness

from src.fairness import (
    base_rates,
    calibration_by_group,
    demographic_parity,
    equalized_odds,
    premium_shift,
    threshold_for_tpr,
)

plt.rcParams["figure.dpi"] = 110
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

names = ["age", "workclass", "fnlwgt", "education", "education_num",
         "marital_status", "occupation", "relationship", "race", "sex",
         "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"]

adult = pd.read_csv(ROOT / "data" / "adult.data", names=names, na_values=" ?", skipinitialspace=True)
df = adult.dropna().copy()
df["income_high"] = (df["income"] == ">50K").astype(int)
print(f"rows after cleaning: {len(df):,} | base income>50K rate: {df['income_high'].mean():.3f}")

rows after cleaning: 32,561 | base income>50K rate: 0.241


## Fit the model

A logistic GLM with logit link — the same GLM machinery as Project 1, just a binomial family instead of Poisson/Gamma.

In [2]:
formula = ("income_high ~ age + education_num + hours_per_week + "
           "C(sex) + C(race) + C(workclass) + C(marital_status) + C(occupation)")
model = smf.glm(formula, data=df, family=sm.families.Binomial())
fit = model.fit()
print(fit.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            income_high   No. Observations:                32561
Model:                            GLM   Df Residuals:                    32525
Model Family:                Binomial   Df Model:                           35
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11565.
Date:                Mon, 10 Aug 2026   Deviance:                       23131.
Time:                        00:43:21   Pearson chi2:                 3.02e+04
No. Iterations:                    22   Pseudo R-squ. (CS):             0.3254
Covariance Type:            nonrobust                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

In [3]:
scored = df.copy()
scored["policy_id"] = np.arange(len(scored))
scored["predicted"] = fit.predict(df)

print(f"Overall mean predicted probability: {scored['predicted'].mean():.4f}")
print(f"Calibration (overall): predicted={scored['predicted'].mean():.4f} vs actual={scored['income_high'].mean():.4f}")

def outcome_rates(groups):
    tab = scored.groupby(list(groups), observed=True).agg(
        n_policies=("policy_id", "count"),
        positive_rate=("income_high", "mean"),
    )
    return tab.round(4)

Overall mean predicted probability: 0.2408
Calibration (overall): predicted=0.2408 vs actual=0.2408


## Base rates by protected group

Precondition check: does the outcome (income > 50K) differ across groups? (Using `claim_rate` as a generic label for the positive-outcome rate.)

In [4]:
outcome_rates(("sex",))

,n_policies,positive_rate
sex,,
Female,10771,0.1095
Male,21790,0.3057


In [5]:
outcome_rates(("race",))

,n_policies,positive_rate
race,,
Amer-Indian-Eskimo,311,0.1158
Asian-Pac-Islander,1039,0.2656
Black,3124,0.1239
Other,271,0.0923
White,27816,0.2559


## Demographic parity

Mean predicted probability per group vs the overall mean. (Same math as premium parity in Project 1.)

In [6]:
demographic_parity(scored, "predicted", ("sex",))

,n_policies,mean_score,ratio_vs_overall,diff_vs_overall
sex,,,,
Female,10771,0.1095,0.4546,-0.1313
Male,21790,0.3057,1.2696,0.0649


In [7]:
demographic_parity(scored, "predicted", ("race",))

,n_policies,mean_score,ratio_vs_overall,diff_vs_overall
race,,,,
Amer-Indian-Eskimo,311,0.1158,0.4807,-0.1251
Asian-Pac-Islander,1039,0.2656,1.1031,0.0248
Black,3124,0.1239,0.5144,-0.1169
Other,271,0.0923,0.3831,-0.1486
White,27816,0.2559,1.0625,0.0151


## Equalized odds

TPR/FPR at a common threshold (predicted probability > 0.5).

In [8]:
equalized_odds(scored, "predicted", "income_high", ("sex",), threshold=0.5)

,n,tpr,fpr,predicted_positive_rate
sex,,,,
Female,10771.0,0.3121,0.0155,0.0480
Male,21790.0,0.5916,0.1146,0.2604


In [9]:
equalized_odds(scored, "predicted", "income_high", ("race",), threshold=0.5)

,n,tpr,fpr,predicted_positive_rate
race,,,,
Amer-Indian-Eskimo,311.0,0.2500,0.0109,0.0386
Asian-Pac-Islander,1039.0,0.6123,0.1206,0.2512
Black,3124.0,0.3618,0.0230,0.0650
Other,271.0,0.2800,0.0163,0.0406
White,27816.0,0.5598,0.0831,0.2051


## Calibration parity

Mean predicted vs mean actual by group — the property the GLM tends to satisfy.

In [10]:
calibration_by_group(scored, "income_high", "predicted", ("sex",))

,n_policies,actual_mean,predicted_mean,predicted/actual
sex,,,,
Female,10771,0.1095,0.1095,1.0
Male,21790,0.3057,0.3057,1.0


In [11]:
calibration_by_group(scored, "income_high", "predicted", ("race",))

,n_policies,actual_mean,predicted_mean,predicted/actual
race,,,,
Amer-Indian-Eskimo,311,0.1158,0.1158,1.0
Asian-Pac-Islander,1039,0.2656,0.2656,1.0
Black,3124,0.1239,0.1239,1.0
Other,271,0.0923,0.0923,1.0
White,27816,0.2559,0.2559,1.0


## The impossibility, reproduced on real data

If base rates differ by sex, the threshold needed to equalize true-positive rates differs — so calibration and equalized odds cannot both hold.

In [12]:
threshold_for_tpr(scored, "predicted", "income_high", ("sex",), target_tpr=0.5)

,needed_threshold,base_rate
sex,,
Female,0.3430,0.1095
Male,0.5892,0.3057


## Prediction shift (who gets flagged most) 

In [13]:
premium_shift(scored, "predicted", ("sex",))

,mean_premium,premium_ratio_vs_cheapest,premium_gap_vs_cheapest
sex,,,
Female,0.11,1.00,0.0
Male,0.31,2.79,0.2


## Save results

In [14]:
outcome_rates(("sex",)).to_csv(RESULTS / "adult_base_rates_sex.csv")
outcome_rates(("race",)).to_csv(RESULTS / "adult_base_rates_race.csv")
demographic_parity(scored, "predicted", ("sex",)).to_csv(RESULTS / "adult_dp_sex.csv")
demographic_parity(scored, "predicted", ("race",)).to_csv(RESULTS / "adult_dp_race.csv")
equalized_odds(scored, "predicted", "income_high", ("sex",), threshold=0.5).to_csv(RESULTS / "adult_eo_sex.csv")
equalized_odds(scored, "predicted", "income_high", ("race",), threshold=0.5).to_csv(RESULTS / "adult_eo_race.csv")
calibration_by_group(scored, "income_high", "predicted", ("sex",)).to_csv(RESULTS / "adult_calibration_sex.csv")
calibration_by_group(scored, "income_high", "predicted", ("race",)).to_csv(RESULTS / "adult_calibration_race.csv")
threshold_for_tpr(scored, "predicted", "income_high", ("sex",), target_tpr=0.5).to_csv(RESULTS / "adult_thresholds_sex.csv")

# Chart: mean predicted probability by sex x race
tab = scored.groupby(["sex", "race"], observed=True)["predicted"].mean().unstack()
fig, ax = plt.subplots(figsize=(9, 4.5))
tab.plot.bar(ax=ax, edgecolor="white")
ax.set_ylabel("Mean predicted P(income>50K)")
ax.set_title("Predicted income probability by sex and race (TEST)")
plt.tight_layout()
plt.savefig(RESULTS / "adult_prediction_by_group.png", dpi=150)
plt.close()
print("saved results to", RESULTS)

saved results to C:\Users\29226\Desktop\Github\EXP-project-1\test_public_examples\results


## Observations (TEST)

* On this data, calibration holds within each group while demographic parity and equalized odds fail — the same pattern as the synthetic flagship.
* The per-group thresholds for TPR parity differ (e.g., by sex), reproducing the Chouldechova conflict on real data.
* **Caveats:** this is a deliberately naive model (protected attributes included, no feature engineering, no train/test split here), and `income > 50K` is a proxy for a "favorable outcome" — the point is to demonstrate the *pipeline*, not to draw policy conclusions.